In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/patientpunk.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


# Judgement ⑨ — side-effect extraction: do models agree on the AE set?

For each (post, drug) the classifier extracts a **set of side-effects**. This is a fully-wired field that already feeds the FDA tolerability analysis, so its reliability matters — but it has no external gold, so the only honest question is *agreement*: given the same post and drug, do models extract the same adverse events? Measured across the roster on the IRR labelled pairs, metric = mean pairwise Jaccard on the AE sets.

In [ ]:

import json, re
import numpy as np
from collections import defaultdict, Counter
from itertools import combinations
d=json.load(open(r"C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/data/validation/j9_sideeffects_runs.json")); R=[r for r in d["results"] if not r["parse_failed"]]
MODELS=sorted(set(r["model"] for r in R)); short=lambda m:m.split("/")[-1]
# per (sample, drug): {model: set(side_effects)}
cell=defaultdict(dict)
for r in R:
    se=set(str(x).lower().strip() for x in (r.get("side_effects") or []) if str(x).strip())
    cell[(r["sample_id"], r["drug"])][r["model"]]=se
def jac(a,b): return len(a&b)/len(a|b) if (a|b) else 1.0
# pairwise model agreement on AE sets (only where BOTH models have a value for that post,drug)
pair=defaultdict(list)
for k,mv in cell.items():
    for a,b in combinations(MODELS,2):
        if a in mv and b in mv: pair[(a,b)].append(jac(mv[a],mv[b]))
allsims=[s for v in pair.values() for s in v]
# how often is the AE set EMPTY? (models often report no AEs)
n_cells=sum(len(mv) for mv in cell.values()); n_empty=sum(1 for mv in cell.values() for s in mv.values() if not s)
display(Markdown(f"*(loaded — {len(R)} classifications, {len(MODELS)} models; overall mean pairwise AE-set "
 f"agreement {np.mean(allsims):.0%}; {n_empty/n_cells:.0%} of extractions report no side-effects)*"))


## 1. How much do models agree on the side-effect set?

Mean pairwise Jaccard on the AE sets, per model against the rest.

In [ ]:

per=[]
for m in MODELS:
    sims=[s for (a,b),v in pair.items() if m in (a,b) for s in v]
    per.append((short(m), np.mean(sims) if sims else np.nan))
per.sort(key=lambda x:-x[1])
fig,ax=plt.subplots(figsize=(8,6))
ax.barh([p[0] for p in per],[p[1]*100 for p in per],color="#2a78d6")
for i,p in enumerate(per): ax.text(p[1]*100+0.5,i,f"{p[1]:.0%}",va="center",fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("mean AE-set agreement with other models (Jaccard)"); ax.set_xlim(0,max(p[1] for p in per)*115)
ax.set_title("Side-effect set agreement per model"); fig.tight_layout(); plt.show()
display(Markdown(f"**Agreement is low ({np.mean(allsims):.0%} overall)** — but AE sets are small and sparse, "
 f"so Jaccard is punishing (one model lists `insomnia`, another `sleep problems`, a third nothing → near-zero "
 f"overlap). Two structural reasons agreement is genuinely hard here: (a) side-effects are only mentioned in "
 f"passing in most posts, so presence is noisy; (b) free-text AE strings aren't normalized. **The takeaway "
 f"isn't a clean number — it's that raw side-effect *sets* are the least reliable field, and any AE analysis "
 f"needs string normalization + a presence threshold before trusting cross-model or cross-community counts.**"))


## 2. What gets reported — and the sparsity problem

In [ ]:

allse=Counter(x for mv in cell.values() for s in mv.values() for x in s)
top=allse.most_common(15)
tb=pd.DataFrame(top, columns=["side-effect (verbatim)","times extracted"])
display(HTML("<b>Most-extracted side-effects (raw strings, unnormalised)</b>"+tb.to_html(index=False)))
display(Markdown(f"**{n_empty/n_cells:.0%} of extractions report NO side-effect at all** — most posts don't "
 f"discuss AEs for the drug in question, so the field is mostly empty and the non-empty entries are a thin, "
 f"noisy signal. The raw strings above also show the normalisation gap (synonyms/spellings uncollapsed), which "
 f"is exactly what drags the Jaccard agreement down and what an ontology pass (MedDRA/UMLS) would fix."))


## 3. The two threats that matter for the FDA use

Side-effects feed the FDA tolerability work (`FDA_analysis/build_ldn_ae.py`,
`ldn_two_community_tolerability.ipynb`), so two known threats are load-bearing:

- **LIST-FANOUT** (`intervention_config.py:164`) — the prompt pins a reported symptom to *every* drug named in
  a stacked post. So on multi-treatment posts, each drug inherits the whole post's AE list, inflating per-drug
  side-effect counts exactly as the group-attribution rule inflates per-drug *positives* (judgement ⑤). Any
  per-drug AE profile from stacked posts is suspect.
- **DR4 (verbosity confound)** — the FDA work compares AE profiles *across communities* (CLH vs Phoenix). If
  one community writes longer posts, its AE lists look richer and the comparison measures verbosity, not
  tolerability. **Run DR4's flatness check (AE-set size vs post length, per community) before trusting any
  cross-community AE comparison.**

Both mean: the side-effect field is usable for *within-post presence* signals, but per-drug and
cross-community AE *counts* need the fan-out correction and the verbosity check first.

## 4. Verdict

In [ ]:

lines=[
 f"- **Side-effect sets are the least-reliable field** — ~{np.mean(allsims):.0%} mean pairwise agreement, and "
 f"~{n_empty/n_cells:.0%} of extractions are empty. Partly Jaccard punishing sparse sets, partly genuine "
 "presence noise + unnormalised strings.",
 "- **Not a clean pass/fail** — with no gold, this is agreement-only, and the low number is as much a metric "
 "and sparsity artifact as a model failure (same lesson as the variable-coding N×N). A MedDRA/UMLS "
 "normalization pass + a presence threshold would lift the usable signal substantially.",
 "- **Two corrections are mandatory before the FDA AE use:** the LIST-FANOUT fan-out (per-drug counts on "
 "stacked posts) and the DR4 verbosity check (cross-community counts). Neither is optional given how AE "
 "profiles are compared downstream.",
 "- **Recommendation:** treat raw AE sets as a *presence* signal only; normalize to an ontology and apply the "
 "two corrections before any per-drug or cross-community tolerability claim.",
]
display(Markdown("\n".join(lines)))


## Limitations

- **No gold** — agreement-only; models agreeing on an AE could share a prompt bias, and a low number is partly
  the sparse-set Jaccard artifact, not proven unreliability.
- **Jaccard on tiny sets is high-variance** — one differing term swings a pair from 1.0 to 0.0; absolute levels
  understate semantic agreement (`insomnia` vs `sleep problems`).
- **IRR labelled pairs only** (116 post×drug), K=1 — no within-model variability here; the across-model number
  is the primary readout.
- Describes agreement of a measurement step, not real adverse-event rates. Not medical advice.

In [ ]:

M=d["manifest"]
prov=pd.DataFrame({"item":["field","models","pairs","metric","threats","skill"],
 "value":["side_effects (classify_batch)", str(len(MODELS)), str(M.get("n_pairs","?")),
          "mean pairwise Jaccard on AE sets", "LIST-FANOUT + DR4 verbosity", "research-assistant v2"]})
display(HTML("<b>Provenance</b>"+prov.to_html(index=False)))
display(HTML('<div style="font-size:1.15em;font-weight:bold;font-style:italic;margin-top:1em">'
             'Describes agreement of a measurement step, not real adverse-event rates. Not medical advice.</div>'))
